# Hyperparameter Tuning — All 5 Experiments

Uses **Optuna** (TPE sampler + MedianPruner) to find optimal hyperparameters for each experiment.

Run each section independently. Best params are saved to `results/tuning/expN_best_params.json` and the final model is retrained with those params.

**Install if needed:**
```bash
pip install optuna
```

In [1]:
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import StandardScaler
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
import json, os, copy, warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

ROOT   = Path('../../').resolve()
DEVICE = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs(ROOT / 'results/tuning', exist_ok=True)
os.makedirs(ROOT / 'models', exist_ok=True)

print(f'Device: {DEVICE}')
print(f'Root:   {ROOT}')

Device: mps
Root:   /Users/shrutisivakumar/Library/CloudStorage/OneDrive-Personal/College Stuff/Sem 6/Projects/NLP/Facebook-Hateful-Memes-Challenge-2020


/Users/shrutisivakumar/Library/CloudStorage/OneDrive-Personal/College Stuff/Sem 6/Projects/NLP/Facebook-Hateful-Memes-Challenge-2020/hateful-memes-venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## Shared Utilities

In [2]:
def make_pos_weight(y_train):
    n_pos = (y_train == 1).sum()
    n_neg = (y_train == 0).sum()
    return torch.tensor([n_neg / n_pos], dtype=torch.float32).to(DEVICE)


def run_epoch_unimodal(loader, model, criterion, optimizer=None):
    """Single-input (image-only or text-only) epoch."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, all_probs, all_labels = 0, [], []
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            if is_train:
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item() * len(y_batch)
            all_probs.extend(torch.sigmoid(logits).detach().cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    auc = roc_auc_score(all_labels, all_probs)
    return total_loss / len(all_labels), auc


def run_epoch_multimodal(loader, model, criterion, optimizer=None):
    """Dual-input (image + text) epoch."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, all_probs, all_labels = 0, [], []
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for x_img, x_text, y in loader:
            x_img, x_text, y = x_img.to(DEVICE), x_text.to(DEVICE), y.to(DEVICE)
            logits = model(x_img, x_text)
            loss = criterion(logits, y)
            if is_train:
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item() * len(y)
            all_probs.extend(torch.sigmoid(logits).detach().cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    auc = roc_auc_score(all_labels, all_probs)
    return total_loss / len(all_labels), auc


def make_unimodal_loaders(X_tr, y_tr, X_dv, y_dv, batch_size=64):
    def _ds(X, y):
        return TensorDataset(torch.tensor(X, dtype=torch.float32),
                             torch.tensor(y, dtype=torch.float32))
    return (DataLoader(_ds(X_tr, y_tr), batch_size=batch_size, shuffle=True),
            DataLoader(_ds(X_dv, y_dv), batch_size=batch_size, shuffle=False))


def make_multimodal_loaders(Xi_tr, Xt_tr, y_tr, Xi_dv, Xt_dv, y_dv, batch_size=64):
    def _ds(Xi, Xt, y):
        return TensorDataset(torch.tensor(Xi, dtype=torch.float32),
                             torch.tensor(Xt, dtype=torch.float32),
                             torch.tensor(y,  dtype=torch.float32))
    return (DataLoader(_ds(Xi_tr, Xt_tr, y_tr), batch_size=batch_size, shuffle=True),
            DataLoader(_ds(Xi_dv, Xt_dv, y_dv), batch_size=batch_size, shuffle=False))


def quick_train(model, train_loader, dev_loader, criterion, lr, weight_decay,
                run_fn, epochs=40, patience=8, trial=None):
    """Train with early stopping. Optuna trial optional for pruning."""
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    best_auc, best_state, patience_ctr = 0.0, None, 0

    for epoch in range(1, epochs + 1):
        run_fn(train_loader, model, criterion, optimizer)
        _, dv_auc = run_fn(dev_loader, model, criterion)
        scheduler.step()

        if dv_auc > best_auc:
            best_auc = dv_auc
            best_state = copy.deepcopy(model.state_dict())
            patience_ctr = 0
        else:
            patience_ctr += 1

        # Optuna pruning
        if trial is not None:
            trial.report(dv_auc, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

        if patience_ctr >= patience:
            break

    model.load_state_dict(best_state)
    return model, best_auc


def save_best_params(exp_name, params, best_auc):
    path = ROOT / f'results/tuning/{exp_name}_best_params.json'
    out = {'experiment': exp_name, 'best_dev_auc': round(best_auc, 4), 'params': params}
    with open(path, 'w') as f:
        json.dump(out, f, indent=2)
    print(f'Saved → {path}')
    return out


print('Utilities loaded.')

Utilities loaded.


---
# EXP 1 — Image-Only: ResNet50 + MLP

**Search space:**
- `hidden1`: 256, 512, 1024
- `hidden2`: 64, 128, 256
- `dropout1`: 0.3 – 0.6
- `dropout2`: 0.2 – 0.5
- `lr`: 1e-4 – 1e-2 (log)
- `weight_decay`: 1e-5 – 1e-2 (log)

In [3]:
# --- Load & normalize ---
IMG_DIR = ROOT / 'artifacts/embeddings/image'

X_tr1 = np.load(IMG_DIR / 'train_resnet50.npy').astype(np.float32)
y_tr1 = np.load(IMG_DIR / 'train_labels.npy')
X_dv1 = np.load(IMG_DIR / 'dev_resnet50.npy').astype(np.float32)
y_dv1 = np.load(IMG_DIR / 'dev_labels.npy')

sc1 = StandardScaler().fit(X_tr1)
X_tr1, X_dv1 = sc1.transform(X_tr1), sc1.transform(X_dv1)

pw1 = make_pos_weight(y_tr1)
print(f'Exp1 data ready. Train={X_tr1.shape}, Dev={X_dv1.shape}')

Exp1 data ready. Train=(8500, 2048), Dev=(500, 2048)


In [4]:
class ImageMLP(nn.Module):
    def __init__(self, input_dim=2048, h1=512, h2=128, d1=0.4, d2=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1), nn.LayerNorm(h1), nn.GELU(), nn.Dropout(d1),
            nn.Linear(h1, h2),        nn.LayerNorm(h2), nn.GELU(), nn.Dropout(d2),
            nn.Linear(h2, 1)
        )
    def forward(self, x): return self.net(x).squeeze(-1)


def exp1_objective(trial):
    h1 = trial.suggest_categorical('hidden1', [256, 512, 1024])
    h2 = trial.suggest_categorical('hidden2', [64, 128, 256])
    d1 = trial.suggest_float('dropout1', 0.3, 0.6)
    d2 = trial.suggest_float('dropout2', 0.2, 0.5)
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    wd = trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True)

    # Constrain: h2 must be < h1
    if h2 >= h1:
        raise optuna.exceptions.TrialPruned()

    model = ImageMLP(input_dim=2048, h1=h1, h2=h2, d1=d1, d2=d2).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw1)
    train_loader, dev_loader = make_unimodal_loaders(X_tr1, y_tr1, X_dv1, y_dv1)
    _, best_auc = quick_train(model, train_loader, dev_loader, criterion,
                               lr, wd, run_epoch_unimodal, trial=trial)
    return best_auc


print('Running Exp 1 tuning (30 trials)...')
study1 = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=5)
)
study1.optimize(exp1_objective, n_trials=30, show_progress_bar=True)

print(f'\nBest AUC : {study1.best_value:.4f}')
print(f'Best params: {study1.best_params}')
save_best_params('exp1', study1.best_params, study1.best_value)

Running Exp 1 tuning (30 trials)...


Best trial: 22. Best value: 0.570056: 100%|██████████| 30/30 [02:57<00:00,  5.91s/it]


Best AUC : 0.5701
Best params: {'hidden1': 512, 'hidden2': 64, 'dropout1': 0.4155152925355866, 'dropout2': 0.30701119091242135, 'lr': 0.0017164557754294728, 'weight_decay': 0.00469522143960661}
Saved → /Users/shrutisivakumar/Library/CloudStorage/OneDrive-Personal/College Stuff/Sem 6/Projects/NLP/Facebook-Hateful-Memes-Challenge-2020/results/tuning/exp1_best_params.json


{'experiment': 'exp1',
 'best_dev_auc': 0.5701,
 'params': {'hidden1': 512,
  'hidden2': 64,
  'dropout1': 0.4155152925355866,
  'dropout2': 0.30701119091242135,
  'lr': 0.0017164557754294728,
  'weight_decay': 0.00469522143960661}}

---
# EXP 2 — Text-Only: DistilBERT + MLP

**Search space:**
- `hidden1`: 128, 256, 512
- `hidden2`: 32, 64, 128
- `dropout1`: 0.2 – 0.5
- `dropout2`: 0.1 – 0.4
- `lr`: 1e-4 – 5e-3 (log)
- `weight_decay`: 1e-5 – 1e-2 (log)

*Smaller space than Exp 1 — text embeddings are richer, need less capacity.*

In [6]:
TEXT_DIR = ROOT / 'artifacts/embeddings/text'

X_tr2 = np.load(TEXT_DIR / 'train_distilbert.npy').astype(np.float32)
y_tr2 = np.load(IMG_DIR  / 'train_labels.npy')
X_dv2 = np.load(TEXT_DIR / 'dev_distilbert.npy').astype(np.float32)
y_dv2 = np.load(IMG_DIR  / 'dev_labels.npy')

sc2 = StandardScaler().fit(X_tr2)
X_tr2, X_dv2 = sc2.transform(X_tr2), sc2.transform(X_dv2)

pw2 = make_pos_weight(y_tr2)
print(f'Exp2 data ready. Train={X_tr2.shape}, Dev={X_dv2.shape}')

Exp2 data ready. Train=(8500, 768), Dev=(500, 768)


In [7]:
class TextMLP(nn.Module):
    def __init__(self, input_dim=768, h1=256, h2=64, d1=0.3, d2=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1), nn.LayerNorm(h1), nn.GELU(), nn.Dropout(d1),
            nn.Linear(h1, h2),        nn.LayerNorm(h2), nn.GELU(), nn.Dropout(d2),
            nn.Linear(h2, 1)
        )
    def forward(self, x): return self.net(x).squeeze(-1)


def exp2_objective(trial):
    h1 = trial.suggest_categorical('hidden1', [128, 256, 512])
    h2 = trial.suggest_categorical('hidden2', [32, 64, 128])
    d1 = trial.suggest_float('dropout1', 0.2, 0.5)
    d2 = trial.suggest_float('dropout2', 0.1, 0.4)
    lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
    wd = trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True)

    if h2 >= h1:
        raise optuna.exceptions.TrialPruned()

    model = TextMLP(input_dim=768, h1=h1, h2=h2, d1=d1, d2=d2).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw2)
    train_loader, dev_loader = make_unimodal_loaders(X_tr2, y_tr2, X_dv2, y_dv2)
    _, best_auc = quick_train(model, train_loader, dev_loader, criterion,
                               lr, wd, run_epoch_unimodal, trial=trial)
    return best_auc


print('Running Exp 2 tuning (30 trials)...')
study2 = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=5)
)
study2.optimize(exp2_objective, n_trials=30, show_progress_bar=True)

print(f'\nBest AUC : {study2.best_value:.4f}')
print(f'Best params: {study2.best_params}')
save_best_params('exp2', study2.best_params, study2.best_value)

Running Exp 2 tuning (30 trials)...


Best trial: 12. Best value: 0.647008: 100%|██████████| 30/30 [02:15<00:00,  4.52s/it]


Best AUC : 0.6470
Best params: {'hidden1': 128, 'hidden2': 64, 'dropout1': 0.49895798560352256, 'dropout2': 0.1952262723429521, 'lr': 0.004906509649994044, 'weight_decay': 0.0012464402007299144}
Saved → /Users/shrutisivakumar/Library/CloudStorage/OneDrive-Personal/College Stuff/Sem 6/Projects/NLP/Facebook-Hateful-Memes-Challenge-2020/results/tuning/exp2_best_params.json


{'experiment': 'exp2',
 'best_dev_auc': 0.647,
 'params': {'hidden1': 128,
  'hidden2': 64,
  'dropout1': 0.49895798560352256,
  'dropout2': 0.1952262723429521,
  'lr': 0.004906509649994044,
  'weight_decay': 0.0012464402007299144}}

---
# EXP 3 — ResNet50 vs ViT + DistilBERT + Proj-Concat + MLP

**Search space (shared for both backbones):**
- `proj_dim`: 128, 256, 512
- `hidden1`: 64, 128, 256
- `hidden2`: 16, 32, 64
- `dropout_proj`: 0.1 – 0.4
- `dropout1`: 0.2 – 0.5
- `dropout2`: 0.1 – 0.4
- `lr`: 1e-4 – 5e-3 (log)
- `weight_decay`: 1e-5 – 1e-2 (log)

*Each backbone gets its own study — best params may differ.*

In [9]:
IMG_DIMS = {'resnet50': 2048, 'vit': 768}

# Load all splits for both backbones
raw3 = {}
for img_key in ['resnet50', 'vit']:
    Xi_tr = np.load(IMG_DIR  / f'train_{img_key}.npy').astype(np.float32)
    Xt_tr = np.load(TEXT_DIR / 'train_distilbert.npy').astype(np.float32)
    y_tr  = np.load(IMG_DIR  / 'train_labels.npy')
    Xi_dv = np.load(IMG_DIR  / f'dev_{img_key}.npy').astype(np.float32)
    Xt_dv = np.load(TEXT_DIR / 'dev_distilbert.npy').astype(np.float32)
    y_dv  = np.load(IMG_DIR  / 'dev_labels.npy')

    sci = StandardScaler().fit(Xi_tr)
    sct = StandardScaler().fit(Xt_tr)
    raw3[img_key] = (
        sci.transform(Xi_tr), sct.transform(Xt_tr), y_tr,
        sci.transform(Xi_dv), sct.transform(Xt_dv), y_dv
    )
    print(f'{img_key}: img={Xi_tr.shape}, text={Xt_tr.shape}')

pw3 = make_pos_weight(raw3['resnet50'][2])

resnet50: img=(8500, 2048), text=(8500, 768)
vit: img=(8500, 768), text=(8500, 768)


In [10]:
class ProjectionFusionMLP(nn.Module):
    def __init__(self, img_dim, text_dim=768, proj_dim=256,
                 d_proj=0.3, h1=128, h2=32, d1=0.3, d2=0.2):
        super().__init__()
        self.img_proj  = nn.Sequential(nn.Linear(img_dim,   proj_dim), nn.GELU(), nn.Dropout(d_proj))
        self.text_proj = nn.Sequential(nn.Linear(text_dim,  proj_dim), nn.GELU(), nn.Dropout(d_proj))
        self.classifier = nn.Sequential(
            nn.Linear(proj_dim * 2, h1), nn.LayerNorm(h1), nn.GELU(), nn.Dropout(d1),
            nn.Linear(h1, h2),           nn.LayerNorm(h2), nn.GELU(), nn.Dropout(d2),
            nn.Linear(h2, 1)
        )
    def forward(self, x_img, x_text):
        return self.classifier(
            torch.cat([self.img_proj(x_img), self.text_proj(x_text)], dim=-1)
        ).squeeze(-1)


def make_exp3_objective(img_key):
    Xi_tr, Xt_tr, y_tr, Xi_dv, Xt_dv, y_dv = raw3[img_key]
    img_dim = IMG_DIMS[img_key]

    def objective(trial):
        proj_dim = trial.suggest_categorical('proj_dim', [128, 256, 512])
        h1       = trial.suggest_categorical('hidden1',  [64, 128, 256])
        h2       = trial.suggest_categorical('hidden2',  [16, 32, 64])
        d_proj   = trial.suggest_float('dropout_proj', 0.1, 0.4)
        d1       = trial.suggest_float('dropout1',     0.2, 0.5)
        d2       = trial.suggest_float('dropout2',     0.1, 0.4)
        lr       = trial.suggest_float('lr',           1e-4, 5e-3, log=True)
        wd       = trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True)

        if h2 >= h1:
            raise optuna.exceptions.TrialPruned()

        model = ProjectionFusionMLP(img_dim=img_dim, proj_dim=proj_dim,
                                     d_proj=d_proj, h1=h1, h2=h2, d1=d1, d2=d2).to(DEVICE)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pw3)
        tr_loader, dv_loader = make_multimodal_loaders(Xi_tr, Xt_tr, y_tr, Xi_dv, Xt_dv, y_dv)
        _, best_auc = quick_train(model, tr_loader, dv_loader, criterion,
                                   lr, wd, run_epoch_multimodal, trial=trial)
        return best_auc
    return objective


studies3 = {}
for img_key in ['resnet50', 'vit']:
    print(f'\nTuning Exp 3 — {img_key.upper()} + DistilBERT (30 trials)...')
    study = optuna.create_study(
        direction='maximize',
        sampler=TPESampler(seed=42),
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=5)
    )
    study.optimize(make_exp3_objective(img_key), n_trials=30, show_progress_bar=True)
    studies3[img_key] = study
    print(f'Best AUC ({img_key}): {study.best_value:.4f}')
    print(f'Best params: {study.best_params}')
    save_best_params(f'exp3_{img_key}', study.best_params, study.best_value)


Tuning Exp 3 — RESNET50 + DistilBERT (30 trials)...


Best trial: 2. Best value: 0.67648: 100%|██████████| 30/30 [03:22<00:00,  6.75s/it]


Best AUC (resnet50): 0.6765
Best params: {'proj_dim': 512, 'hidden1': 256, 'hidden2': 16, 'dropout_proj': 0.12930163420191518, 'dropout1': 0.4052699079536471, 'dropout2': 0.23204574812188042, 'lr': 0.000161190447276092, 'weight_decay': 0.0003058656666978527}
Saved → /Users/shrutisivakumar/Library/CloudStorage/OneDrive-Personal/College Stuff/Sem 6/Projects/NLP/Facebook-Hateful-Memes-Challenge-2020/results/tuning/exp3_resnet50_best_params.json

Tuning Exp 3 — VIT + DistilBERT (30 trials)...


Best trial: 25. Best value: 0.68128: 100%|██████████| 30/30 [03:46<00:00,  7.56s/it] 

Best AUC (vit): 0.6813
Best params: {'proj_dim': 512, 'hidden1': 64, 'hidden2': 16, 'dropout_proj': 0.3119673709739203, 'dropout1': 0.3231939855156805, 'dropout2': 0.17305300348035824, 'lr': 0.0002623436007711755, 'weight_decay': 1.1443137160728232e-05}
Saved → /Users/shrutisivakumar/Library/CloudStorage/OneDrive-Personal/College Stuff/Sem 6/Projects/NLP/Facebook-Hateful-Memes-Challenge-2020/results/tuning/exp3_vit_best_params.json


---
# EXP 4 — Global vs YOLO (Best Backbone from Exp 3)

Same search space as Exp 3. Best backbone auto-loaded from `results/exp3.json`.

In [12]:
with open(ROOT / 'results/exp3.json') as f:
    exp3_meta = json.load(f)

BEST_BB  = exp3_meta['best_image_backbone']
IMG_DIM4 = exp3_meta['best_img_dim']
print(f'Best backbone: {BEST_BB.upper()} (dim={IMG_DIM4})')

raw4 = {}
for variant, img_key in [('global', BEST_BB), ('yolo', f'{BEST_BB}_yolo')]:
    Xi_tr = np.load(IMG_DIR  / f'train_{img_key}.npy').astype(np.float32)
    Xt_tr = np.load(TEXT_DIR / 'train_distilbert.npy').astype(np.float32)
    y_tr  = np.load(IMG_DIR  / 'train_labels.npy')
    Xi_dv = np.load(IMG_DIR  / f'dev_{img_key}.npy').astype(np.float32)
    Xt_dv = np.load(TEXT_DIR / 'dev_distilbert.npy').astype(np.float32)
    y_dv  = np.load(IMG_DIR  / 'dev_labels.npy')

    sci = StandardScaler().fit(Xi_tr)
    sct = StandardScaler().fit(Xt_tr)
    raw4[variant] = (
        sci.transform(Xi_tr), sct.transform(Xt_tr), y_tr,
        sci.transform(Xi_dv), sct.transform(Xt_dv), y_dv
    )
    print(f'[{variant}] img={Xi_tr.shape}')

pw4 = make_pos_weight(raw4['global'][2])

Best backbone: VIT (dim=768)
[global] img=(8500, 768)
[yolo] img=(8500, 768)


In [13]:
def make_exp4_objective(variant):
    Xi_tr, Xt_tr, y_tr, Xi_dv, Xt_dv, y_dv = raw4[variant]

    def objective(trial):
        proj_dim = trial.suggest_categorical('proj_dim', [128, 256, 512])
        h1       = trial.suggest_categorical('hidden1',  [64, 128, 256])
        h2       = trial.suggest_categorical('hidden2',  [16, 32, 64])
        d_proj   = trial.suggest_float('dropout_proj', 0.1, 0.4)
        d1       = trial.suggest_float('dropout1',     0.2, 0.5)
        d2       = trial.suggest_float('dropout2',     0.1, 0.4)
        lr       = trial.suggest_float('lr',           1e-4, 5e-3, log=True)
        wd       = trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True)

        if h2 >= h1:
            raise optuna.exceptions.TrialPruned()

        model = ProjectionFusionMLP(img_dim=IMG_DIM4, proj_dim=proj_dim,
                                     d_proj=d_proj, h1=h1, h2=h2, d1=d1, d2=d2).to(DEVICE)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pw4)
        tr_loader, dv_loader = make_multimodal_loaders(Xi_tr, Xt_tr, y_tr, Xi_dv, Xt_dv, y_dv)
        _, best_auc = quick_train(model, tr_loader, dv_loader, criterion,
                                   lr, wd, run_epoch_multimodal, trial=trial)
        return best_auc
    return objective


studies4 = {}
for variant in ['global', 'yolo']:
    print(f'\nTuning Exp 4 — {BEST_BB.upper()} [{variant.upper()}] + DistilBERT (30 trials)...')
    study = optuna.create_study(
        direction='maximize',
        sampler=TPESampler(seed=42),
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=5)
    )
    study.optimize(make_exp4_objective(variant), n_trials=30, show_progress_bar=True)
    studies4[variant] = study
    print(f'Best AUC ({variant}): {study.best_value:.4f}')
    save_best_params(f'exp4_{variant}', study.best_params, study.best_value)


Tuning Exp 4 — VIT [GLOBAL] + DistilBERT (30 trials)...


Best trial: 21. Best value: 0.673936: 100%|██████████| 30/30 [03:21<00:00,  6.71s/it]


Best AUC (global): 0.6739
Saved → /Users/shrutisivakumar/Library/CloudStorage/OneDrive-Personal/College Stuff/Sem 6/Projects/NLP/Facebook-Hateful-Memes-Challenge-2020/results/tuning/exp4_global_best_params.json

Tuning Exp 4 — VIT [YOLO] + DistilBERT (30 trials)...


Best trial: 18. Best value: 0.672768: 100%|██████████| 30/30 [04:50<00:00,  9.69s/it]

Best AUC (yolo): 0.6728
Saved → /Users/shrutisivakumar/Library/CloudStorage/OneDrive-Personal/College Stuff/Sem 6/Projects/NLP/Facebook-Hateful-Memes-Challenge-2020/results/tuning/exp4_yolo_best_params.json


---
# EXP 5 — Text Backbone: DistilBERT vs RoBERTa (Best Image from Exp 4)

**RoBERTa underperformed in untuned run (0.6364 vs 0.6633).**  
This tuning run targets that gap specifically by expanding the LR search range upward — RoBERTa embeddings often need a higher LR to push the projection layer into a useful subspace.

**Extra search space for RoBERTa:**
- `lr`: 1e-4 – 1e-2 (wider than Exp 3/4)
- `proj_dim`: 128, 256, 512 (larger proj may help RoBERTa)

DistilBERT result reused from Exp 4 (no retraining).

In [18]:
with open(ROOT / 'results/exp4.json') as f:
    exp4_meta = json.load(f)

BEST_IMG_KEY = exp4_meta['best_image_key']
IMG_DIM5     = exp4_meta['img_dim']
best_repr = exp4_meta['best_image_representation']
DISTILBERT_AUC5 = exp4_meta['results'][best_repr]['dev_auc_roc']

print(f'Best image key : {BEST_IMG_KEY} (dim={IMG_DIM5})')
print(f'DistilBERT AUC : {DISTILBERT_AUC5:.4f}  (reused from Exp 4)')

Xi_tr5 = np.load(IMG_DIR  / f'train_{BEST_IMG_KEY}.npy').astype(np.float32)
Xt_tr5 = np.load(TEXT_DIR / 'train_roberta.npy').astype(np.float32)
y_tr5  = np.load(IMG_DIR  / 'train_labels.npy')
Xi_dv5 = np.load(IMG_DIR  / f'dev_{BEST_IMG_KEY}.npy').astype(np.float32)
Xt_dv5 = np.load(TEXT_DIR / 'dev_roberta.npy').astype(np.float32)
y_dv5  = np.load(IMG_DIR  / 'dev_labels.npy')

sci5 = StandardScaler().fit(Xi_tr5)
sct5 = StandardScaler().fit(Xt_tr5)
Xi_tr5, Xi_dv5 = sci5.transform(Xi_tr5), sci5.transform(Xi_dv5)
Xt_tr5, Xt_dv5 = sct5.transform(Xt_tr5), sct5.transform(Xt_dv5)

pw5 = make_pos_weight(y_tr5)
print(f'Data loaded: img={Xi_tr5.shape}, text(RoBERTa)={Xt_tr5.shape}')

Best image key : vit_yolo (dim=768)
DistilBERT AUC : 0.6636  (reused from Exp 4)
Data loaded: img=(8500, 768), text(RoBERTa)=(8500, 768)


In [19]:
def exp5_objective(trial):
    proj_dim = trial.suggest_categorical('proj_dim', [128, 256, 512])
    h1       = trial.suggest_categorical('hidden1',  [64, 128, 256])
    h2       = trial.suggest_categorical('hidden2',  [16, 32, 64])
    d_proj   = trial.suggest_float('dropout_proj', 0.1, 0.4)
    d1       = trial.suggest_float('dropout1',     0.2, 0.5)
    d2       = trial.suggest_float('dropout2',     0.1, 0.4)
    lr       = trial.suggest_float('lr',           1e-4, 1e-2, log=True)  # wider for RoBERTa
    wd       = trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True)

    if h2 >= h1:
        raise optuna.exceptions.TrialPruned()

    model = ProjectionFusionMLP(img_dim=IMG_DIM5, proj_dim=proj_dim,
                                 d_proj=d_proj, h1=h1, h2=h2, d1=d1, d2=d2).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw5)
    tr_loader, dv_loader = make_multimodal_loaders(Xi_tr5, Xt_tr5, y_tr5,
                                                    Xi_dv5, Xt_dv5, y_dv5)
    _, best_auc = quick_train(model, tr_loader, dv_loader, criterion,
                               lr, wd, run_epoch_multimodal, trial=trial)
    return best_auc


print('Tuning Exp 5 — RoBERTa (30 trials, wider LR range)...')
study5 = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=5)
)
study5.optimize(exp5_objective, n_trials=30, show_progress_bar=True)

print(f'\nBest AUC (RoBERTa tuned) : {study5.best_value:.4f}')
print(f'DistilBERT AUC (Exp 4)   : {DISTILBERT_AUC5:.4f}')
print(f'Delta                    : {study5.best_value - DISTILBERT_AUC5:+.4f}')
print(f'Best params: {study5.best_params}')
save_best_params('exp5_roberta', study5.best_params, study5.best_value)

Tuning Exp 5 — RoBERTa (30 trials, wider LR range)...


Best trial: 18. Best value: 0.67696: 100%|██████████| 30/30 [05:21<00:00, 10.71s/it]


Best AUC (RoBERTa tuned) : 0.6770
DistilBERT AUC (Exp 4)   : 0.6636
Delta                    : +0.0134
Best params: {'proj_dim': 512, 'hidden1': 128, 'hidden2': 64, 'dropout_proj': 0.34990176943796875, 'dropout1': 0.3844441061352564, 'dropout2': 0.34419811167334946, 'lr': 0.005634783190720708, 'weight_decay': 0.0001863574604726976}
Saved → /Users/shrutisivakumar/Library/CloudStorage/OneDrive-Personal/College Stuff/Sem 6/Projects/NLP/Facebook-Hateful-Memes-Challenge-2020/results/tuning/exp5_roberta_best_params.json


{'experiment': 'exp5_roberta',
 'best_dev_auc': 0.677,
 'params': {'proj_dim': 512,
  'hidden1': 128,
  'hidden2': 64,
  'dropout_proj': 0.34990176943796875,
  'dropout1': 0.3844441061352564,
  'dropout2': 0.34419811167334946,
  'lr': 0.005634783190720708,
  'weight_decay': 0.0001863574604726976}}